In [ ]:
import math
import time

import gradio as gr
import gradio as gr
import pandas as pd
import plotly.express as px
from datasets import load_dataset, Audio

from sdoml_task1.features import build_feature_dataset
from sdoml_task1.modeling.load import load_features
from sdoml_task1.modeling.train import run_training
from sdoml_task1.modeling.optimization import OPTIMIZERS, SCHEDULERS, REGULARIZATIONS, ACTIVATIONS

X_train, y_train, X_test, y_test = load_features()

In [ ]:
MAX_LAYERS = 10

def update_slider_visibility(n_layers):
    """Show only the first ``n_layers`` hidden-layer inputs."""
    updates = []
    for i in range(MAX_LAYERS):
        if i < n_layers:
            updates.append(gr.Number(visible=True))
        else:
            updates.append(gr.Number(visible=False))
    return updates


def gradio_train(epochs, lr_text, batch_size, n_layers, activation_function, regularization,
                 reg_param,  optimizer_name, scheduler_name, limit_epoch, *layer_sizes):
    """Read the values from the interface, train the model and return a summary."""
    
    try:
        lr = float(lr_text)
    except ValueError:
        raise gr.Error(f"The learning rate must be a number (e.g. 0.001, got '{lr_text}'.")
    if lr <= 0:
        raise gr.Error("The learning rate must be positive.")

    hidden_sizes = []
    for i in range(n_layers):
        if layer_sizes[i] is None or layer_sizes[i] < 1:
            raise gr.Error(f"Hidden layer {i+1} must have at least 1 neuron.")
        hidden_sizes.append(layer_sizes[i])


    if regularization == "None":
        regularization = None
        
    if regularization == "dropout":
        if reg_param < 0 or reg_param >= 1:
            raise gr.Error("For dropout, the value is between 0 and 1.")

    metrics = run_training(X_train, y_train, X_test, y_test, hidden_sizes=hidden_sizes,
                           activation_function=activation_function, regularization=regularization,
                           reg_param=reg_param, optimizer_name=optimizer_name, scheduler_name=scheduler_name,
                           epochs=int(epochs), lr=lr, batch_size=int(batch_size), limit_epoch=int(limit_epoch))

    epochs_done = len(metrics["loss_val"])
    final_acc = metrics["accuracy_val"][-1]
    best_acc = max(metrics["accuracy_val"])

    summary = (
        f"**Modele trained:**   \n"
        f"**Final validation accuracy:** {final_acc:.4f}    \n"
        f"**Best validation accuracy:** {best_acc:.4f}    \n"
        f"**Final validation loss:** {metrics['loss_val'][-1]:.4f}"
    )
    return summary

### Remember to close the port with .close() or by restarting the kernel. If you don't, it sometimes struggles to re-run the code cause the port is busy.

In [ ]:
def create_app():
    with gr.Blocks() as interface:
        gr.Markdown(r"""
            # Spoken Digit Classification
            ### Project description
            This project uses the [Audio MNIST Dataset](https://huggingface.co/datasets/gilkeyio/AudioMNIST) to train a model that predicts numbers from 0 to 9 based on audio input. The neural network is a multilayer perceptron with an input of 26 neurons, two hidden layers of 64 and 32 neurons, and a final output layer of 10 neurons. It uses the Adam optimizer and the cross-entropy loss function.
        """)

        with gr.Tabs():
            # ========================================= DATA EXPLORATION TAB =========================================
            with gr.Tab("Data"):
                gr.Markdown("The dataset has 50 English recordings per digit (0-9) of 60 speakers. There are 60 participants in total, with 12 being women and 48 being men, all featuring a diverse range of accents and country of origin. Their ages vary from 22 to 61 years old. This is a great dataset to explore a simple audio classification problem: either the digit or the gender.")
                with gr.Group():
                    gr.Markdown("Below you can ispect the dataset. Choose the filters you want and plot the data.")
    
                    with gr.Accordion("Filters", open=False):
                        with gr.Row():
                            digit_filter = gr.Dropdown(
                                choices=filter_options["digit"],
                                value="All",
                                label="Digit"
                            )
                            speaker_id_filter = gr.Dropdown(
                                choices=filter_options["speaker_id"],
                                value="All",
                                label="Speaker ID"
                            )
                            gender_filter = gr.Dropdown(
                                choices=filter_options["gender"],
                                value="All",
                                label="Gender"
                            )
                        
                        with gr.Row():
                            accent_filter = gr.Dropdown(
                                choices=filter_options["accent"],
                                value="All",
                                label="Accent"
                            )
                            age_filter = gr.Dropdown(
                                choices=filter_options["age"],
                                value="All",
                                label="Age"
                            )
                            origin_filter = gr.Dropdown(
                                choices=filter_options["origin"],
                                value="All",
                                label="Origin"
                            )
                        
                        with gr.Row():
                            split_filter = gr.Dropdown(
                                choices=filter_options["split"],
                                value="All",
                                label="Split"
                            )

                        # Reset button
                        with gr.Row():
                            reset_btn = gr.Button("🔄 Reset Filters", variant="secondary")

                    with gr.Accordion("Data table", open=False):
                        # Data table output
                        data_table = gr.Dataframe(
                            value=df,
                            interactive=False,
                            label="Data"
                        )
                        
                        # Result count
                        result_text = gr.Textbox(
                            value=f"Showing {len(df)} entries",
                            interactive=False,
                            label="Result Count"
                        )

                    # Create event listeners for each filter
                    filters = [digit_filter, speaker_id_filter, gender_filter, accent_filter, age_filter, origin_filter, split_filter]

                    with gr.Accordion("Plots", open=True):
                        plot_column = gr.Dropdown(
                            choices=["Digit", "Speaker ID", "Gender", "Accent", "Age", "Origin", "Split"],
                            value="Digit",
                            label="Select Column to Plot"
                        )
                        
                        plots_output = gr.Plot()
                        
                        # Update plot when filters change
                        for filter_elem in filters:
                            filter_elem.change(
                                fn=update_plot,
                                inputs=filters + [plot_column],
                                outputs=[plots_output]
                            )
                        
                        # Update plot when dropdown changes
                        plot_column.change(
                            fn=update_plot,
                            inputs=filters + [plot_column],
                            outputs=[plots_output]
                        )
                    
                    for filter_elem in filters:
                        filter_elem.change(
                            fn=filter_data,
                            inputs=filters,
                            outputs=[data_table, result_text]
                        )
                

            # ======================================== TRAINING INTERFACE TAB ========================================

            with gr.Tab("Train"):
                gr.Markdown("#### Training Interface")

                gr.Markdown("Choose the settings, then click **Train**. Note that if the number of epoch is high, it may take lot of time")
                with gr.Row():
                    with gr.Column(scale=2):
                        with gr.Row():
                            epochs_input = gr.Number(value=10, minimum=1, precision=0, label="Epochs")
                            lr_input = gr.Textbox(value="0.001", label="Learning rate")
                            batch_input = gr.Number(value=64, minimum=1, precision=0, label="Batch size")

                        n_layers_slider = gr.Slider(minimum=1, maximum=MAX_LAYERS, value=1, step=1, label="Number of hidden layers")
                        
                        with gr.Row():
                            layer_inputs = []
                            for i in range(MAX_LAYERS):
                                layer_input = gr.Number(value=64, minimum=1, precision=0, label=f"Layer {i+1} neurons", visible=(i == 0))
                                layer_inputs.append(layer_input)

                        activation = gr.Radio(ACTIVATIONS, value="relu", label="Activation function")
                        optimizer = gr.Radio(OPTIMIZERS, value="Adam", label="Optimizer")
                        scheduler = gr.Radio(SCHEDULERS, value="None", label="Learning rate scheduler")
                        regularization = gr.Radio(REGULARIZATIONS, value="None", label="Regularization")
                        
                        with gr.Row():
                            reg_param_input = gr.Number(value=0.0, minimum=0, label="Regularization strength", info="L1 / weight decay / dropout probability")
                            patience_input = gr.Number(value=10, minimum=1, precision=0, label="Early stopping patience", info="Only early stopping")

                        train_button = gr.Button("Train", variant="primary")

                    
                    with gr.Column(scale=1):
                        summary_output = gr.Markdown("*Results will appear here after training.*")

               
                n_layers_slider.change(fn=update_slider_visibility, inputs=n_layers_slider, outputs=layer_inputs)

                train_inputs = [epochs_input, lr_input, batch_input, n_layers_slider, activation, regularization, reg_param_input,
                                optimizer, scheduler, patience_input, *layer_inputs]
                train_button.click(fn=gradio_train, inputs=train_inputs, outputs=summary_output)



            # ========================================= MODEL EVALUATION TAB =========================================
            with gr.Tab("Model"):
                gr.Markdown("#### Model Evaluation")

        gr.Markdown(r"""
            *Created by: Amets Martiarena, Teva Philippe, Alvaro Crespo | Check the original repo [here](https://github.com/AmetsMarti/SDOML-project1)*.
        """)
    return interface

In [ ]:
try:
    interface.close()
except:
    pass

# Set to true to publish the app
send_it = False

interface = create_app()
interface.launch(server_port=7860, share=send_it)

In [ ]:
interface.close()